# tokenizer.py, in cells

Copied verbatim from `cs336_basics/tokenizer.py`. Nothing fixed, nothing added.
Run the cells top to bottom; the last one still raises `KeyError`, same as the script.

In [ ]:
import regex as re

In [ ]:
def get_stats(ids)-> dict: # this is just 
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

In [ ]:
def merge(ids, pair, idx):
    out = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(idx)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

In [ ]:
pattern = re.compile("|".join([
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]*[\p{Ll}\p{Lm}\p{Lo}\p{M}]+(?i:'s|'t|'re|'ve|'m|'ll|'d)?""",
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]+[\p{Ll}\p{Lm}\p{Lo}\p{M}]*(?i:'s|'t|'re|'ve|'m|'ll|'d)?""",
    r"""\p{N}{1,3}""",
    r""" ?[^\s\p{L}\p{N}]+[\r\n/]*""",
    r"""\s*[\r\n]+""",
    r"""\s+(?!\S)""",
    r"""\s+""",
]))

In [ ]:
def train(text, vocab_size, verbose=False):

    #merges: list[tuple[bytes, bytes]]
    text = pattern.findall(text) # this will return a list of string 
    ids = list(chunk.encode("utf-8") for chunk in text)
    vocab = {i: bytes([i]) for i in range(256)}
    # 257 : b "cd" 
    merges_list = []
    merges = {} # (97,98) -> 257 

    for idx in range(256, vocab_size):
        stats = get_stats(ids)
        if not stats:                                # nothing left to merge
            break
        pair = max(stats, key=lambda p: (stats[p], vocab[p[0]], vocab[p[1]]), )        # tie breaking base on the bytes if want to go ofr the lower use - ? 
        ids = merge(ids, pair, idx)
        merges[pair] = idx
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
        merges_list.append((vocab[pair[0]], vocab[pair[1]]))
        if verbose:
            print(f"merge {idx - 255}/{vocab_size - 256}: {pair} -> {idx} {vocab[idx]} ({stats[pair]}x)")

    return vocab, merges_list

In [ ]:
def encode(text):
    pass

def decode():
    pass

## Run it

In [ ]:
text = "cd cd abab"
vocab, merges = train(text, 269)
print(pattern.findall("cd cd abab"))
print(len(merges))

## Scratch

Poke at the pieces separately here. Things worth looking at on their own:

- `pattern.findall(text)` — the pre-tokens
- what each pre-token looks like after `.encode("utf-8")`, and what `list()` of that gives
- `get_stats(...)` on a single pre-token's byte list
- the frequency table: distinct pre-token -> how many times it appears